## Imports

In [85]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    PowerTransformer,
    OrdinalEncoder,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

## Data

In [86]:
X = pd.read_csv("smartphone_battery_features.csv")
y = pd.read_csv("smartphone_battery_targets.csv")

print(X.shape)
print(y.shape)

(5000, 15)
(5000, 3)


In [87]:
df = pd.merge(X, y, on="Device_ID")

df.head()

,Device_ID,device_age_months,battery_capacity_mah,avg_screen_on_hours_per_day,avg_charging_cycles_per_week,avg_battery_temp_celsius,fast_charging_usage_percent,overnight_charging_freq_per_week,gaming_hours_per_week,video_streaming_hours_per_week,background_app_usage_level,signal_strength_avg,charging_habit_score,usage_intensity_score,thermal_stress_index,current_battery_health_percent,recommended_action
0,207dd94c-0430-43aa-b388-4893447e628e,38,4500,7.1,11.4,34.8,90.8,7,7.9,14.0,Medium,Poor,4,10.0,4.04,32.8,Change Phone
1,3f4d1d33-ba89-4814-a168-7b4cc75be26b,28,3000,6.8,10.3,35.4,60.6,2,8.6,11.0,Medium,Good,7,10.0,4.23,50.3,Replace Battery
2,b4adca05-564f-4b70-ab69-e8d66e656463,14,3000,7.2,11.2,29.4,29.3,4,0.3,10.3,Medium,Good,6,10.0,2.21,66.1,Replace Battery
3,4147e039-31b7-480a-bbc9-03cd0f66e9f1,42,3000,5.5,8.3,32.8,62.5,0,1.9,4.9,Medium,Good,8,10.0,3.13,46.8,Change Phone
4,3f9b0fb7-73c2-4ab7-8e30-7b492097a3f5,7,3000,7.6,11.6,38.7,85.4,6,7.9,9.3,High,Good,5,10.0,4.95,67.2,Replace Battery


In [88]:
drop_cols = [
    "thermal_stress_index",
    "avg_charging_cycles_per_week",
    "charging_habit_score",
    "Device_ID"
]

df = df.drop(columns=drop_cols)

In [89]:
y_reg = df["current_battery_health_percent"]

label_encoder = LabelEncoder()
y_clf = label_encoder.fit_transform(df["recommended_action"])

X = df.drop(
    columns=[
        "current_battery_health_percent",
        "recommended_action"
    ]
)

## split

In [90]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

In [91]:
numeric_cols = X.select_dtypes(include="number").columns
categorical_cols = X.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

## reg model

In [92]:
from sklearn.ensemble import GradientBoostingRegressor

reg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

reg_model.fit(X_train_r, y_train_r)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [93]:
pred = reg_model.predict(X_test_r)

mae = mean_absolute_error(y_test_r, pred)
rmse = np.sqrt(mean_squared_error(y_test_r, pred))
r2 = r2_score(y_test_r, pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 3.285933354462162
RMSE: 4.172614274952063
R2: 0.9443937077276937


## clf model

In [94]:
from sklearn.ensemble import GradientBoostingClassifier

clf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        random_state=42
    ))
])

clf_model.fit(X_train_c, y_train_c)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [96]:
pred = clf_model.predict(X_test_c)

acc = accuracy_score(y_test_c, pred)
f1 = f1_score(y_test_c, pred, average="weighted")

print("Accuracy:", acc)
print("F1 score:", f1)

Accuracy: 0.871
F1 score: 0.8711316467559695


## compare several models

In [95]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train_r, y_train_r)

    pred = pipe.predict(X_test_r)

    results.append([
        name,
        r2_score(y_test_r, pred)
    ])

pd.DataFrame(results, columns=["Model", "R2"])

,Model,R2
0,Linear,0.914075
1,Ridge,0.914073
2,Decision Tree,0.871052
3,Random Forest,0.940472
4,Gradient Boosting,0.944394


In [97]:
!pip install mlflow


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [101]:
models = {
    "Linear": LinearRegression(),
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor()
}

In [105]:
import mlflow

mlflow.set_experiment("Battery Health Regression")

2026/06/04 22:10:06 INFO mlflow.tracking.fluent: Experiment with name 'Battery Health Regression' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1780593006770, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1780593006770, lifecycle_stage='active', name='Battery Health Regression', tags={}, trace_location=None, workspace='default'>

In [106]:
for name, model in models.items():

    with mlflow.start_run(run_name=name):

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipe.fit(X_train_r, y_train_r)

        pred = pipe.predict(X_test_r)

        r2 = r2_score(y_test_r, pred)

        mlflow.log_param("model", name)
        mlflow.log_metric("r2", r2)

        mlflow.sklearn.log_model(
            pipe,
            "model"
        )

2026/06/04 22:10:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 22:10:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Linear at: http://127.0.0.1:5000/#/experiments/2/runs/d699bfd10f3041f0a91680f5290cd1a9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/06/04 22:10:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 22:10:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/bfd3b39bf5bc4110b6ad84aa5490a305
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/06/04 22:10:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 22:10:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/2/runs/41107d89f9ba4323948c7f02c322895a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [107]:
for exp in mlflow.search_experiments():
    print(exp.name)

Battery Health Regression
genai-retention-regression
Default
